# Data Processing: Employee Performance Analysis, INX Future Inc.

**Purpose of this notebook:** take the original, untouched employee data dump, check its quality, clean it, create a few additional (engineered) features, and save the final data sets used for exploration and modelling.

**Steps**
1. Load the raw data and the data definitions
2. Check data quality (missing values, duplicates, invalid values, outliers)
3. Clean the data
4. Engineer additional features
5. Encode categorical variables for modelling
6. Save the processed data sets

> The raw file is never modified. Everything created here is written to `data/processed/`.

In [1]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 60, 'display.width', 200)

ROOT = Path('../..').resolve()
RAW = ROOT/'data'/'raw'
PROCESSED = ROOT/'data'/'processed'
REFERENCES = ROOT/'references'
PROCESSED.mkdir(parents=True, exist_ok=True)
RAW_FILE = RAW/'INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1_8.xls'
print('Raw file found:', RAW_FILE.exists())

Raw file found: True


## 1. Load the raw data and the data definitions

In [2]:
# The workbook has two sheets: the employee records and the data definitions.
# (Reading .xls files needs the 'xlrd' package:  pip install xlrd)
raw = pd.read_excel(RAW_FILE, sheet_name=0, engine='xlrd')
definitions = pd.read_excel(RAW_FILE, sheet_name=1, engine='xlrd', header=None)
print('Rows, columns:', raw.shape)
raw.head()

Rows, columns:

 (1200, 28)


,EmpNumber,Age,Gender,EducationBackground,MaritalStatus,EmpDepartment,EmpJobRole,BusinessTravelFrequency,DistanceFromHome,EmpEducationLevel,EmpEnvironmentSatisfaction,EmpHourlyRate,EmpJobInvolvement,EmpJobLevel,EmpJobSatisfaction,NumCompaniesWorked,OverTime,EmpLastSalaryHikePercent,EmpRelationshipSatisfaction,TotalWorkExperienceInYears,TrainingTimesLastYear,EmpWorkLifeBalance,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition,PerformanceRating
0,E1001000,32,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,10,3,4,55,3,2,4,1,No,12,4,10,2,2,10,7,0,8,No,3
1,E1001006,47,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,14,4,4,42,3,2,1,2,No,12,4,20,2,3,7,7,1,7,No,3
2,E1001007,40,Male,Life Sciences,Married,Sales,Sales Executive,Travel_Frequently,5,4,4,48,2,3,1,5,Yes,21,3,20,2,3,18,13,1,12,No,4
3,E1001009,41,Male,Human Resources,Divorced,Human Resources,Manager,Travel_Rarely,10,4,2,73,2,5,4,3,No,15,2,23,2,2,21,6,12,6,No,3
4,E1001010,60,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,16,4,1,84,3,2,1,8,No,14,4,10,1,3,2,2,2,2,No,3


In [3]:
# Save the data definitions as a readable reference document
defs = definitions.dropna(how='all').copy()
defs.columns = ['Variable', 'Meaning of coded values']
defs['Variable'] = defs['Variable'].ffill()
defs.to_csv(REFERENCES/'data_definitions.csv', index=False)
defs

,Variable,Meaning of coded values
2,EmpEducationLevel,1 'Below College'
3,EmpEducationLevel,2 'College'
4,EmpEducationLevel,3 'Bachelor'
5,EmpEducationLevel,4 'Master'
6,EmpEducationLevel,5 'Doctor'
8,EmpEnvironmentSatisfaction,1 'Low'
9,EmpEnvironmentSatisfaction,2 'Medium'
10,EmpEnvironmentSatisfaction,3 'High'
11,EmpEnvironmentSatisfaction,4 'Very High'
13,EmpJobInvolvement,1 'Low'


## 2. Data quality checks

We check for: missing values, duplicate rows, duplicate employee numbers, values outside the documented range, inconsistent text and unusual values (outliers).

In [4]:
quality = pd.DataFrame({
    'data_type': raw.dtypes.astype(str),
    'missing_values': raw.isna().sum(),
    'unique_values': raw.nunique()
})
print('Total missing values  :', int(raw.isna().sum().sum()))
print('Duplicate rows        :', int(raw.duplicated().sum()))
print('Duplicate employee IDs:', int(raw['EmpNumber'].duplicated().sum()))
quality

Total missing values  : 0
Duplicate rows        : 0
Duplicate employee IDs: 0


,data_type,missing_values,unique_values
EmpNumber,str,0,1200
Age,int64,0,43
Gender,str,0,2
EducationBackground,str,0,6
MaritalStatus,str,0,3
EmpDepartment,str,0,6
EmpJobRole,str,0,19
BusinessTravelFrequency,str,0,3
DistanceFromHome,int64,0,29
EmpEducationLevel,int64,0,5


In [5]:
# Values must stay inside the documented ranges (rating scales run from 1 to 4, education from 1 to 5)
scale_1_4 = ['EmpEnvironmentSatisfaction','EmpJobInvolvement','EmpJobSatisfaction',
             'EmpRelationshipSatisfaction','EmpWorkLifeBalance','PerformanceRating']
problems = {c: int((~raw[c].between(1,4)).sum()) for c in scale_1_4}
problems['EmpEducationLevel'] = int((~raw['EmpEducationLevel'].between(1,5)).sum())
print('Rows outside the documented range:', problems)

# Logical consistency: time spent in a role / with a manager cannot exceed time at the company, which cannot exceed total experience
checks = {
  'years in role > years at company'      : (raw.ExperienceYearsInCurrentRole > raw.ExperienceYearsAtThisCompany).sum(),
  'years with manager > years at company' : (raw.YearsWithCurrManager > raw.ExperienceYearsAtThisCompany).sum(),
  'years at company > total experience'   : (raw.ExperienceYearsAtThisCompany > raw.TotalWorkExperienceInYears).sum(),
  'age < 18'                              : (raw.Age < 18).sum(),
}
pd.Series(checks, name='rows violating the rule')

Rows outside the documented range: {'EmpEnvironmentSatisfaction': 0, 'EmpJobInvolvement': 0, 'EmpJobSatisfaction': 0, 'EmpRelationshipSatisfaction': 0, 'EmpWorkLifeBalance': 0, 'PerformanceRating': 0, 'EmpEducationLevel': 0}


years in role > years at company         0
years with manager > years at company    0
years at company > total experience      0
age < 18                                 0
Name: rows violating the rule, dtype: int64

In [6]:
# Text columns: look for stray spaces or inconsistent spelling
text_cols = raw.select_dtypes(exclude='number').columns.drop('EmpNumber')
for c in text_cols:
    print(f'{c:26s}', sorted(raw[c].unique()))

Gender                     ['Female', 'Male']
EducationBackground        ['Human Resources', 'Life Sciences', 'Marketing', 'Medical', 'Other', 'Technical Degree']
MaritalStatus              ['Divorced', 'Married', 'Single']
EmpDepartment              ['Data Science', 'Development', 'Finance', 'Human Resources', 'Research & Development', 'Sales']
EmpJobRole                 ['Business Analyst', 'Data Scientist', 'Delivery Manager', 'Developer', 'Finance Manager', 'Healthcare Representative', 'Human Resources', 'Laboratory Technician', 'Manager', 'Manager R&D', 'Manufacturing Director', 'Research Director', 'Research Scientist', 'Sales Executive', 'Sales Representative', 'Senior Developer', 'Senior Manager R&D', 'Technical Architect', 'Technical Lead']
BusinessTravelFrequency    ['Non-Travel', 'Travel_Frequently', 'Travel_Rarely']
OverTime                   ['No', 'Yes']
Attrition                  ['No', 'Yes']


In [7]:
# Outliers: count values beyond 1.5 x interquartile range for the continuous variables
continuous = ['Age','DistanceFromHome','EmpHourlyRate','NumCompaniesWorked','EmpLastSalaryHikePercent',
              'TotalWorkExperienceInYears','TrainingTimesLastYear','ExperienceYearsAtThisCompany',
              'ExperienceYearsInCurrentRole','YearsSinceLastPromotion','YearsWithCurrManager']
rows = []
for c in continuous:
    q1, q3 = raw[c].quantile([.25,.75]); iqr = q3-q1
    n = int(((raw[c] < q1-1.5*iqr) | (raw[c] > q3+1.5*iqr)).sum())
    rows.append((c, raw[c].min(), raw[c].max(), n, round(100*n/len(raw),1)))
pd.DataFrame(rows, columns=['variable','min','max','outlier_count','outlier_%'])

,variable,min,max,outlier_count,outlier_%
0,Age,18,60,0,0.0
1,DistanceFromHome,1,29,0,0.0
2,EmpHourlyRate,30,100,0,0.0
3,NumCompaniesWorked,0,9,39,3.2
4,EmpLastSalaryHikePercent,11,25,0,0.0
5,TotalWorkExperienceInYears,0,40,51,4.2
6,TrainingTimesLastYear,0,6,188,15.7
7,ExperienceYearsAtThisCompany,0,40,56,4.7
8,ExperienceYearsInCurrentRole,0,18,16,1.3
9,YearsSinceLastPromotion,0,15,88,7.3


**Findings from the quality checks**

- No missing values, no duplicate rows and no duplicate employee numbers.
- All coded values are inside their documented ranges and the tenure variables are logically consistent.
- A few variables (years since last promotion, total experience, years at company) have some high values, but they are genuine long-serving employees, not data errors. They are **kept**; the tree-based models we use are not sensitive to them.

Because the data is already clean, cleaning is limited to tidying text and fixing data types.

## 3. Clean the data

In [8]:
clean = raw.copy()
for c in clean.select_dtypes(exclude='number').columns:
    clean[c] = clean[c].astype(str).str.strip()
clean = clean.drop_duplicates().reset_index(drop=True)

# Readable labels for the performance rating (from the data definitions)
rating_labels = {1:'Low', 2:'Good', 3:'Excellent', 4:'Outstanding'}
clean['PerformanceLabel'] = clean['PerformanceRating'].map(rating_labels)
print(clean['PerformanceRating'].value_counts().sort_index())

PerformanceRating
2    194
3    874
4    132
Name: count, dtype: int64


**Important observation about the target variable.** The definitions list four rating levels (1 Low, 2 Good, 3 Excellent, 4 Outstanding), but **no employee in this data has a rating of 1**. The lowest rating actually present is **2 (Good)** (about 16% of employees). In this project, "lower-performing" therefore means rating 2, and "top performing" means rating 4.

## 4. Feature engineering

Three ratio features are created. They capture ideas that the raw columns express only indirectly:

| New feature | Formula | Why it may matter |
|---|---|---|
| `PromotionDelayRatio` | years since last promotion / (years at company + 1) | How much of an employee's time at the company has passed without a promotion |
| `RoleStagnationRatio` | years in current role / (years at company + 1) | Whether the employee has stayed in the same role for most of their career here |
| `AvgYearsPerCompany` | total experience / (number of companies worked + 1) | A proxy for job-hopping versus stability |

The "+ 1" avoids division by zero for new joiners.

In [9]:
def add_engineered_features(df):
    out = df.copy()
    out['PromotionDelayRatio'] = out['YearsSinceLastPromotion'] / (out['ExperienceYearsAtThisCompany'] + 1)
    out['RoleStagnationRatio'] = out['ExperienceYearsInCurrentRole'] / (out['ExperienceYearsAtThisCompany'] + 1)
    out['AvgYearsPerCompany']  = out['TotalWorkExperienceInYears'] / (out['NumCompaniesWorked'] + 1)
    return out

clean = add_engineered_features(clean)
clean[['PromotionDelayRatio','RoleStagnationRatio','AvgYearsPerCompany']].describe().round(2)

,PromotionDelayRatio,RoleStagnationRatio,AvgYearsPerCompany
count,1200.00,1200.00,1200.00
mean,0.24,0.49,4.24
std,0.27,0.27,4.15
min,0.00,0.00,0.00
25%,0.00,0.33,1.66
50%,0.14,0.50,3.00
75%,0.43,0.67,5.00
max,0.92,0.88,38.00


## 5. Encode categorical variables for modelling

Machine-learning models need numbers. We use:
- **Binary encoding (0/1)** for `Gender`, `OverTime`, `Attrition`
- **Ordinal encoding** for `BusinessTravelFrequency` (Non-Travel < Travel_Rarely < Travel_Frequently), since the categories have a natural order
- **One-hot encoding** for `EducationBackground`, `MaritalStatus`, `EmpDepartment`, `EmpJobRole` (no natural order)

The employee number is an identifier, not a predictor, so it is excluded from the modelling table.

In [10]:
model_ready = clean.drop(columns=['EmpNumber','PerformanceLabel']).copy()
model_ready['Gender']   = (model_ready['Gender'] == 'Male').astype(int)
model_ready['OverTime'] = (model_ready['OverTime'] == 'Yes').astype(int)
model_ready['Attrition']= (model_ready['Attrition'] == 'Yes').astype(int)
model_ready['BusinessTravelFrequency'] = model_ready['BusinessTravelFrequency'].map(
    {'Non-Travel':0, 'Travel_Rarely':1, 'Travel_Frequently':2})
model_ready = pd.get_dummies(model_ready, columns=['EducationBackground','MaritalStatus','EmpDepartment','EmpJobRole'], dtype=int)
print('Model-ready table:', model_ready.shape)
model_ready.head(3)

Model-ready table: (1200, 60)


,Age,Gender,BusinessTravelFrequency,DistanceFromHome,EmpEducationLevel,EmpEnvironmentSatisfaction,EmpHourlyRate,EmpJobInvolvement,EmpJobLevel,EmpJobSatisfaction,NumCompaniesWorked,OverTime,EmpLastSalaryHikePercent,EmpRelationshipSatisfaction,TotalWorkExperienceInYears,TrainingTimesLastYear,EmpWorkLifeBalance,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition,PerformanceRating,PromotionDelayRatio,RoleStagnationRatio,AvgYearsPerCompany,EducationBackground_Human Resources,EducationBackground_Life Sciences,EducationBackground_Marketing,EducationBackground_Medical,EducationBackground_Other,EducationBackground_Technical Degree,MaritalStatus_Divorced,MaritalStatus_Married,MaritalStatus_Single,EmpDepartment_Data Science,EmpDepartment_Development,EmpDepartment_Finance,EmpDepartment_Human Resources,EmpDepartment_Research & Development,EmpDepartment_Sales,EmpJobRole_Business Analyst,EmpJobRole_Data Scientist,EmpJobRole_Delivery Manager,EmpJobRole_Developer,EmpJobRole_Finance Manager,EmpJobRole_Healthcare Representative,EmpJobRole_Human Resources,EmpJobRole_Laboratory Technician,EmpJobRole_Manager,EmpJobRole_Manager R&D,EmpJobRole_Manufacturing Director,EmpJobRole_Research Director,EmpJobRole_Research Scientist,EmpJobRole_Sales Executive,EmpJobRole_Sales Representative,EmpJobRole_Senior Developer,EmpJobRole_Senior Manager R&D,EmpJobRole_Technical Architect,EmpJobRole_Technical Lead
0,32,1,1,10,3,4,55,3,2,4,1,0,12,4,10,2,2,10,7,0,8,0,3,0.000000,0.636364,5.000000,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,47,1,1,14,4,4,42,3,2,1,2,0,12,4,20,2,3,7,7,1,7,0,3,0.125000,0.875000,6.666667,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
2,40,1,2,5,4,4,48,2,3,1,5,1,21,3,20,2,3,18,13,1,12,0,4,0.052632,0.684211,3.333333,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0


## 6. Save the processed data sets

In [11]:
clean.to_csv(PROCESSED/'employee_clean.csv', index=False)
model_ready.to_csv(PROCESSED/'employee_model_ready.csv', index=False)
print('Saved:', [p.name for p in PROCESSED.iterdir()])

Saved: ['employee_model_ready.csv', 'employee_clean.csv']


**Output of this notebook**
- `data/processed/employee_clean.csv`: cleaned data with readable categories plus the engineered features (used for exploration and for the final model pipeline, which does its own encoding)
- `data/processed/employee_model_ready.csv`: fully numeric version (used for correlation analysis)
- `references/data_definitions.csv`: the coded-value dictionary